In [1]:
import pandas as pd
from sklearn.datasets import load_diabetes

# Cargar dataset
diabetes = load_diabetes(as_frame=True)
df = diabetes.frame

# Ver primeras filas
df.head()

,age,sex,bmi,bp,s1,s2,s3,s4,s5,s6,target
0,0.038076,0.050680,0.061696,0.021872,-0.044223,-0.034821,-0.043401,-0.002592,0.019907,-0.017646,151.0
1,-0.001882,-0.044642,-0.051474,-0.026328,-0.008449,-0.019163,0.074412,-0.039493,-0.068332,-0.092204,75.0
2,0.085299,0.050680,0.044451,-0.005670,-0.045599,-0.034194,-0.032356,-0.002592,0.002861,-0.025930,141.0
3,-0.089063,-0.044642,-0.011595,-0.036656,0.012191,0.024991,-0.036038,0.034309,0.022688,-0.009362,206.0
4,0.005383,-0.044642,-0.036385,0.021872,0.003935,0.015596,0.008142,-0.002592,-0.031988,-0.046641,135.0


In [2]:
# Dimensiones
print(df.shape)

# Estadísticas generales
print(df.describe())

# Comprobar nulos
print(df.isnull().sum())

(442, 11)
                age           sex           bmi            bp            s1  \
count  4.420000e+02  4.420000e+02  4.420000e+02  4.420000e+02  4.420000e+02   
mean  -2.511817e-19  1.230790e-17 -2.245564e-16 -4.797570e-17 -1.381499e-17   
std    4.761905e-02  4.761905e-02  4.761905e-02  4.761905e-02  4.761905e-02   
min   -1.072256e-01 -4.464164e-02 -9.027530e-02 -1.123988e-01 -1.267807e-01   
25%   -3.729927e-02 -4.464164e-02 -3.422907e-02 -3.665608e-02 -3.424784e-02   
50%    5.383060e-03 -4.464164e-02 -7.283766e-03 -5.670422e-03 -4.320866e-03   
75%    3.807591e-02  5.068012e-02  3.124802e-02  3.564379e-02  2.835801e-02   
max    1.107267e-01  5.068012e-02  1.705552e-01  1.320436e-01  1.539137e-01   

                 s2            s3            s4            s5            s6  \
count  4.420000e+02  4.420000e+02  4.420000e+02  4.420000e+02  4.420000e+02   
mean   3.918434e-17 -5.777179e-18 -9.042540e-18  9.268604e-17  1.130318e-17   
std    4.761905e-02  4.761905e-02  4.7619

In [3]:
X = df.drop(columns=["target"])
y = df["target"]

In [4]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [5]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [11]:
import mlflow
import mlflow.sklearn
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error, explained_variance_score, max_error
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Activar autologging de scikit-learn (registra muchas cosas automáticamente)
mlflow.sklearn.autolog(log_models=True, log_input_examples=True, log_datasets=True)

# Convertir datos a numpy arrays para evitar warnings de 'Series' flatten
if isinstance(y_train, pd.Series):
    y_train = y_train.to_numpy()
if isinstance(y_test, pd.Series):
    y_test = y_test.to_numpy()
if isinstance(X_train, pd.DataFrame):
    X_train = X_train.to_numpy()
if isinstance(X_test, pd.DataFrame):
    X_test = X_test.to_numpy()

# Iniciar el run con nombre legible
with mlflow.start_run(run_name="RandomForest_Baseline_Fixed_v3"):

    # Hiperparámetros
    params = {
        "n_estimators": 100,
        "max_depth": 5,
        "min_samples_split": 2,
        "min_samples_leaf": 1,
        "random_state": 42
    }
    
    # Loguear parámetros
    mlflow.log_params(params)

    # Crear y entrenar el modelo
    model = RandomForestRegressor(**params)
    model.fit(X_train, y_train)

    # Predicciones
    y_pred = model.predict(X_test)
    y_pred_train = model.predict(X_train)

    # Calcular métricas ampliadas
    metrics = {
        "mse": mean_squared_error(y_test, y_pred),
        "rmse": np.sqrt(mean_squared_error(y_test, y_pred)),
        "mae": mean_absolute_error(y_test, y_pred),
        "r2": r2_score(y_test, y_pred),
        "explained_variance": explained_variance_score(y_test, y_pred),
        "max_error": max_error(y_test, y_pred),
        # Métricas en train para detectar overfitting
        "train_mse": mean_squared_error(y_train, y_pred_train),
        "train_r2": r2_score(y_train, y_pred_train),
    }
    
    # Loguear todas las métricas
    mlflow.log_metrics(metrics)

    # Guardar el modelo (versión correcta para MLflow 3.10+)
    mlflow.sklearn.log_model(
        sk_model=model,
        name="random_forest_model"          # Solo 'name', sin 'artifact_path'
    )

    # ── Generar y loguear gráficos como artefactos ──
    
    # 1. Predicciones vs valores reales
    plt.figure(figsize=(8, 6))
    plt.scatter(y_test, y_pred, alpha=0.5)
    plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2)
    plt.xlabel("Valores reales")
    plt.ylabel("Predicciones")
    plt.title("Predicciones vs Valores Reales")
    plt.tight_layout()
    plt.savefig("pred_vs_real.png")
    mlflow.log_artifact("pred_vs_real.png", artifact_path="plots")
    plt.close()

    # 2. Importancia de características (top 10)
    importances = model.feature_importances_
    sorted_idx = np.argsort(importances)[::-1]
    
    plt.figure(figsize=(10, 6))
    n_features = min(10, len(importances))
    plt.barh(range(n_features), importances[sorted_idx[:n_features]], align='center')
    plt.yticks(range(n_features), [f"Feature {i}" for i in sorted_idx[:n_features]])
    plt.xlabel("Importancia")
    plt.title("Top 10 Features más importantes")
    plt.tight_layout()
    plt.savefig("feature_importance.png")
    mlflow.log_artifact("feature_importance.png", artifact_path="plots")
    plt.close()

    # Imprimir resumen para verificar
    print("Métricas registradas:")
    for k, v in metrics.items():
        print(f"  {k}: {v:.4f}")
    print("\n¡Run completado! Revisa la UI de MLflow → Overview, Model metrics y Artifacts")

2026/02/21 22:52:49 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/02/21 22:52:57 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Métricas registradas:
  mse: 2885.5907
  rmse: 53.7177
  mae: 43.6095
  r2: 0.4554
  explained_variance: 0.4554
  max_error: 144.3543
  train_mse: 1575.5230
  train_r2: 0.7407

¡Run completado! Revisa la UI de MLflow → Overview, Model metrics y Artifacts
